### Over-representation analysis for fetal reversion using gseapy

In [1]:
import gseapy as gp
from gseapy import barplot, dotplot
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pickle

In [2]:
ora_results_dir = "overrepresentation_analysis_plots/"
os.makedirs(ora_results_dir, exist_ok=True)

In [3]:
disease_contrast = 'disease_binary_Y_vs_N'
fetal_contrast = 'age_group_fetal_vs_young'

Examine intersection of cell types for which there is strong fetal reversion and for which there are enough DEGs/DARs for each of the different disease subtypes, and perform ORA 

- Cardiomyocyte, Endothelial, Epicardial, Fibroblast, Myeloid, Neuronal, Pericyte, vSMC

In [35]:
fetalization_cell_types = ["Cardiomyocyte", "Endothelial", "Fibroblast", 
                           "LEC", "Myeloid", "Neuronal", "Pericyte"]

In [12]:
def obtain_ORA_results(gene_list, gmt_file):
    '''
    Run over-representation analysis (ORA) for a list of genes
    '''

    # don't run if the gene_list is empty
    if len(gene_list) == 0:
        return None

    enr = gp.enrichr(gene_list=gene_list, # or "./tests/data/gene_list.txt",
                 gene_sets=gmt_file,
                 organism='human', # don't forget to set organism to the one you desired! e.g. Yeast
                 outdir=None, # don't write to disk
                )

    # extract the results as a df
    ora_res_df = enr.res2d
    # sort by the adjusted p-value
    ora_res_df = ora_res_df.sort_values(by = "Adjusted P-value")

    return(ora_res_df)

In [25]:
def run_ORA_analysis(cell_type, gene_set_gmt, log2FC_threshold = 0.5, p_adj_threshold=0.05):
    '''For a particular cell type, load in the fetal and disease DEGs, and run ORA for those genes that are 
    up in both and down in both. 

    Inputs: 
    - cell_type: The cell type for which to perform the analysis (loads in the results_dict) 
    - gene_set_gmt: The path to the gene sets to search for overrepresentation against
    - log2FC threshold and p_adj_threshold for DEGs 
    '''
    # open the results dictionary
    fetal_results_df = pd.read_csv("../developmental_DEG_analysis/pydeseq2_results/" + cell_type + "_age_group_fetal_vs_young_results.csv",
                                  index_col = 0)
    DCM_results_df = pd.read_csv("../disease_DEG_analysis/non_binarized/pydeseq2_results/" + cell_type + "_disease_DCM_vs_ND_results.csv",
                                    index_col = 0)
    HCM_results_df = pd.read_csv("../disease_DEG_analysis/non_binarized/pydeseq2_results/" + cell_type + "_disease_DCM_vs_ND_results.csv",
                                    index_col = 0)
    ICM_results_df = pd.read_csv("../disease_DEG_analysis/non_binarized/pydeseq2_results/" + cell_type + "_disease_DCM_vs_ND_results.csv",
                                    index_col = 0)

    disease_dict = {
        "DCM": DCM_results_df,
        "HCM": HCM_results_df,
        "ICM": ICM_results_df
    }

    # get fetal up and down genes
    up_in_fetal = fetal_results_df[
        (fetal_results_df['log2FoldChange'] > log2FC_threshold) & 
        (fetal_results_df['padj'] < p_adj_threshold)
    ].index

    down_in_fetal = fetal_results_df[
        (fetal_results_df['log2FoldChange'] < -log2FC_threshold) & 
        (fetal_results_df['padj'] < p_adj_threshold)
    ].index

    # store ORA results for each disease
    up_ora_dict = {}
    down_ora_dict = {}

    for disease_name, disease_df in disease_dict.items():
        # disease up and down genes
        up_in_disease = disease_df[
            (disease_df['log2FoldChange'] > log2FC_threshold) &
            (disease_df['padj'] < p_adj_threshold)
        ].index

        down_in_disease = disease_df[
            (disease_df['log2FoldChange'] < -log2FC_threshold) &
            (disease_df['padj'] < p_adj_threshold)
        ].index

        # intersect with fetal DEGs
        intersecting_up = list(set(up_in_fetal) & set(up_in_disease))
        intersecting_down = list(set(down_in_fetal) & set(down_in_disease))

        # run ORA
        up_ora_res_df = obtain_ORA_results(gene_list=intersecting_up, gmt_file=gene_set_gmt)
        down_ora_res_df = obtain_ORA_results(gene_list=intersecting_down, gmt_file=gene_set_gmt)

        up_ora_dict[disease_name] = up_ora_res_df
        down_ora_dict[disease_name] = down_ora_res_df

    return up_ora_dict, down_ora_dict

In [26]:
hallmark_gmt = "MSigDB_Hallmark_2020"

In [27]:
# test this for one cell type
up_ora_res_df, down_ora_res_df = run_ORA_analysis(cell_type = "Cardiomyocyte", 
                                                  gene_set_gmt=hallmark_gmt)

In [31]:
up_ora_res_df

{'DCM':                 Gene_set                               Term Overlap   P-value  \
 0   MSigDB_Hallmark_2020                        E2F Targets  11/200  0.000438   
 1   MSigDB_Hallmark_2020                    Apical Junction  10/200  0.001623   
 2   MSigDB_Hallmark_2020                            Hypoxia   9/200  0.005461   
 3   MSigDB_Hallmark_2020                    G2-M Checkpoint   9/200  0.005461   
 4   MSigDB_Hallmark_2020                     UV Response Dn   7/144  0.009143   
 5   MSigDB_Hallmark_2020  Epithelial Mesenchymal Transition   8/200  0.016554   
 6   MSigDB_Hallmark_2020                    Mitotic Spindle   7/199  0.043829   
 7   MSigDB_Hallmark_2020            Estrogen Response Early   7/200  0.044818   
 8   MSigDB_Hallmark_2020             Estrogen Response Late   7/200  0.044818   
 9   MSigDB_Hallmark_2020               IL-2/STAT5 Signaling   6/199  0.105425   
 10  MSigDB_Hallmark_2020                         Glycolysis   5/200  0.224712   
 11  MSig

### Now, run for all of the cell types showing fetal reversion

In [36]:
for cell_type in fetalization_cell_types:
    print(cell_type, flush=True)

    # run ORA
    up_ora_dict, down_ora_dict = run_ORA_analysis(cell_type=cell_type, gene_set_gmt=hallmark_gmt)

    # combine upregulated ORA results into one DataFrame with disease column
    up_ora_res_df = pd.concat(
        [df.assign(disease=disease) for disease, df in up_ora_dict.items()],
        ignore_index=True
    )
    up_ora_res_df['regulation'] = 'up'
    up_ora_res_df.to_csv(ora_results_dir + cell_type + "_fetal_disease_up_ORA.csv", index=False)

    # combine downregulated ORA results into one DataFrame with disease column
    down_ora_res_df = pd.concat(
        [df.assign(disease=disease) for disease, df in down_ora_dict.items()],
        ignore_index=True
    )
    down_ora_res_df['regulation'] = 'down'
    down_ora_res_df.to_csv(ora_results_dir + cell_type + "_fetal_disease_down_ORA.csv", index=False)

Cardiomyocyte
Endothelial
Fibroblast
LEC
Myeloid
Neuronal
Pericyte
